In [1]:
import lightgbm as lgb
import numpy as np
import optuna
import polars as pl
from sklearn.model_selection import train_test_split
from surrogate_model.metrics import enrichment_factor, spearman_corr, top_k_recall
from surrogate_model.optuna import RECALL_TOP_1, make_objective

In [2]:
FEATURES = "data/candidates.10k.parquet"
LABELS = "data/1L83.1L83:p2rank:2.10k.parquet"

RANDOM_SEED = 1000
SAMPLE_SIZE = 10000

In [3]:
features = pl.read_parquet(FEATURES)

features = features.filter(
    (pl.col("parse_ok"))
    & (pl.col("error") == "SUCCESS")
    & (pl.col("conversion_error") == "SUCCESS")
)

cns_mpo_schema = pl.Struct(
    [
        pl.Field("clogp", pl.Float64),
        pl.Field("clogd", pl.Float64),
        pl.Field("tpsa", pl.Float64),
    ]
)

features = features.with_columns(
    pl.col("cns_mpo_components").str.json_decode(cns_mpo_schema)
).unnest("cns_mpo_components")

In [4]:
features = features.drop(
    ["smiles", "parse_ok", "pains_flags", "error", "conversion_error"]
)

In [6]:
labels = pl.read_parquet(LABELS)
labels = labels["catalog_id", "affinity_kcal_mol"]

In [7]:
df = features.join(labels, on="catalog_id", how="inner")

In [8]:
if SAMPLE_SIZE < len(df):
    df = df.sample(SAMPLE_SIZE, seed=RANDOM_SEED)

In [9]:
FEATURE_NAMES = ["heavy_atom_count", "molecular_weight", "clogp", "clogd", "tpsa"]

LABEL_NAME = "affinity_kcal_mol"

In [10]:
x = df.select(FEATURE_NAMES).to_numpy()
x = np.hstack([x, np.array(df["morgan_fp"].to_list())])

In [11]:
y = df[LABEL_NAME].to_numpy()

In [12]:
X_train, X_test, y_train, y_test = train_test_split(
    x, y, test_size=0.2, random_state=RANDOM_SEED
)

In [13]:
study = optuna.create_study(direction="minimize")

[I 2026-08-14 23:03:31,531] A new study created in memory with name: no-name-fc72f72b-c535-4213-9080-637c985ebfff


In [ ]:
study.optimize(
    make_objective(
        X_train, y_train, 5, primary_metric=RECALL_TOP_1, random_seed=RANDOM_SEED
    ),
    n_trials=10,
    show_progress_bar=True,
)

  0%|          | 0/10 [00:00<?, ?it/s]

[I 2026-08-14 23:04:15,596] Trial 0 finished with value: 0.1375 and parameters: {'num_leaves': 207, 'max_depth': 5, 'learning_rate': 0.09330555015839914, 'n_estimators': 1151, 'min_child_samples': 76, 'subsample': 0.7207326922769394, 'colsample_bytree': 0.5023273718412065, 'reg_alpha': 0.0008712894482762983, 'reg_lambda': 7.506088681935529e-07}. Best is trial 0 with value: 0.1375.
[I 2026-08-14 23:07:05,581] Trial 1 finished with value: 0.275 and parameters: {'num_leaves': 148, 'max_depth': 9, 'learning_rate': 0.013315395603103172, 'n_estimators': 985, 'min_child_samples': 49, 'subsample': 0.7996709833478286, 'colsample_bytree': 0.5523625659382101, 'reg_alpha': 0.05942145568589688, 'reg_lambda': 0.1680082284501912}. Best is trial 0 with value: 0.1375.
[I 2026-08-14 23:08:20,497] Trial 2 finished with value: 0.325 and parameters: {'num_leaves': 237, 'max_depth': 6, 'learning_rate': 0.013397251009749556, 'n_estimators': 670, 'min_child_samples': 34, 'subsample': 0.58281645329974, 'colsam

In [ ]:
best_params = study.best_params
best_params.update({"random_state": RANDOM_SEED})

final_model = lgb.LGBMRegressor(**best_params)
final_model.fit(
    X_train,
    y_train,
    eval_X=X_test,
    eval_y=y_test,
    callbacks=[lgb.early_stopping(stopping_rounds=50, verbose=True)],
)

In [ ]:
y_pred = final_model.predict(X_test)

In [ ]:
results = {
    "top_1_percent": top_k_recall(y_test, y_pred, 0.01),
    "top_5_percent": top_k_recall(y_test, y_pred, 0.05),
    "top_10_percent": top_k_recall(y_test, y_pred, 0.1),
    "spearman": spearman_corr(y_test, y_pred),
    "enrichment_factor": enrichment_factor(y_test, y_pred, 0.05),
}

In [ ]:
results

# Results

| dataset size | trials | top_1_percent |top_5_percent| top_10_percent | spearman | enrichment |
| - | - | - | - | - | - | - |
| 1000 | 10 | 0.0 | 0.2 | 0.4 | 0.81 | 4.0 |
| 2500 | 10 | 0.2 | 0.48 | 0.66 | 0.83 | 9.6 |
| 10000 
